In [8]:
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# !pip install -q langchain-openai langchain-core requests langchain-groq

^C



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_openai import ChatOpenAI
from langchain_groq.chat_models import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [11]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [12]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [13]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1793059201,
 'time_last_update_utc': 'Tue, 27 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1793145601,
 'time_next_update_utc': 'Wed, 28 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.2718}

In [14]:
convert.invoke({"base_currency_value": 10, "conversion_rate": 95.2718})

952.718

In [15]:

from langchain_google_genai import ChatGoogleGenerativeAI

In [16]:
#Tool buinding

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

In [17]:
model_with_tool=llm.bind_tools([get_conversion_factor,convert])

In [18]:
query = HumanMessage(
    "Call BOTH tools in this turn. "
    "1. Call get_conversion_factor(base_currency='INR', target_currency='USD'). "
    "2. Simultaneously call convert(base_currency_value=10, conversion_rate=1.0) as a placeholder."
)

In [19]:
message=[]

In [20]:
message.append(query)

In [21]:
message

[HumanMessage(content="Call BOTH tools in this turn. 1. Call get_conversion_factor(base_currency='INR', target_currency='USD'). 2. Simultaneously call convert(base_currency_value=10, conversion_rate=1.0) as a placeholder.", additional_kwargs={}, response_metadata={})]

In [ ]:
#Tool caling
result=model_with_tool.invoke(message)

In [ ]:
result.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'fc_c43a87bf-e86e-4c4c-9740-f0ef6efebb7b',
  'type': 'tool_call'}]